# Hospital-Acquired Pressure Injury: PSI-03 and the HAC Reduction Program

Reproduces every figure in [`README.md`](README.md) from CMS public data.

**Author** Yuxuan Huang, Vantara Medical Equipment (Mahoraga Vantara LLC), New York
**Retrieved** 2026-09-17
**Runtime** about five minutes, mostly download

All source data is US Government work under 17 U.S.C. §105.

---

### Disclosure

The author distributes a powered patient transfer system to US healthcare
facilities. That is how the author came to this data. No product is named,
evaluated, or recommended here, and no claim is made that any device affects
the measures described.

### What this analysis does not do

It describes a distribution. It is cross-sectional, has no comparison group,
and evaluates no intervention. Pressure injury arises from nutrition,
repositioning frequency, support surface, perfusion, immobility duration, and
mechanical forces during repositioning and transfer. Nothing here isolates any
one of them.


---
## 1. Environment


In [ ]:
!pip install -q duckdb pandas requests

import duckdb, pandas as pd, requests, os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True)

con = duckdb.connect(':memory:')
def q(sql): return con.execute(sql).df()

print('duckdb', duckdb.__version__)


---
## 2. Resolve current download URLs

CMS publishes a DCAT catalog listing every provider dataset with its current
CSV download URL. The file URLs contain a content hash and change when CMS
republishes; the catalog does not. Resolving at runtime rather than hard-coding
means this notebook keeps working after CMS refreshes the files.


In [ ]:
CATALOG = 'https://data.cms.gov/provider-data/data.json'

cat = requests.get(CATALOG, timeout=180).json()
items = cat.get('dataset', cat)

rows = []
for d in items:
    urls = [x.get('downloadURL') for x in d.get('distribution', [])
            if str(x.get('mediaType','')).endswith('csv')
            or str(x.get('downloadURL','')).lower().endswith('.csv')]
    rows.append({'title': d.get('title'),
                 'modified': d.get('modified'),
                 'url': urls[0] if urls else None})
catalog = pd.DataFrame(rows)

WANT = {
    'complications': 'Complications and Deaths',
    'hac'          : 'Hospital-Acquired Condition',
    'hosp_general' : 'Hospital General Information',
}

resolved = {}
for local, kw in WANT.items():
    m = catalog[catalog.title.str.contains(kw, case=False, na=False) & catalog.url.notna()]
    m = m[~m.title.str.contains('Veterans|VA ', case=False, na=False)]
    resolved[local] = m.iloc[0].url
    print(f'{local:<15} {m.iloc[0].title[:50]:<52} released {m.iloc[0].modified}')


---
## 3. Download


In [ ]:
for local, url in resolved.items():
    path = f'data/{local}.csv'
    r = requests.get(url, timeout=1800, stream=True)
    r.raise_for_status()
    with open(path, 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    print(f'{local:<15} {os.path.getsize(path)/1e6:>7.1f} MB')


---
## 4. Load

Every column is read as text and cast later with `TRY_CAST`, so CMS's
suppressed values and footnote markers become null rather than failing the
load.


In [ ]:
for table, path in [('raw_complications', 'data/complications.csv'),
                    ('raw_hac',           'data/hac.csv'),
                    ('raw_hosp_general',  'data/hosp_general.csv')]:
    con.execute(f"""CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_csv_auto('{path}', header=true, all_varchar=true,
                                    ignore_errors=true, sample_size=-1)""")
    n = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    c = len(con.execute(f'DESCRIBE {table}').df())
    print(f'{table:<20} {n:>8,} rows  {c:>3} cols')


---
## 5. Which measures does CMS publish?

The Complications file is long format: one row per hospital per measure.


In [ ]:
q('''SELECT "Measure ID" AS measure_id,
            ANY_VALUE("Measure Name") AS measure_name,
            COUNT(*) AS n_rows
     FROM raw_complications
     GROUP BY 1 ORDER BY 1''')


PSI-03 is published as its own measure, not only inside the PSI-90
composite. AHRQ defines it as:

> Stage III or IV pressure ulcers or unstageable (secondary diagnosis) per
> 1,000 discharges among surgical or medical patients ages 18 years and older.
> Excludes stays less than 3 days; cases with a principal stage III or IV (or
> unstageable) pressure ulcer diagnosis; cases with a secondary diagnosis of
> stage III or IV pressure ulcer (or unstageable) that is present on admission;
> obstetric cases; severe burns; exfoliative skin disorders.

The present-on-admission exclusion is what makes PSI-03 a measure of
hospital-acquired injury rather than prevalence.


---
## 6. Reshape to one row per hospital


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE hosp_psi AS
SELECT "Facility ID" AS ccn,
  ANY_VALUE("Facility Name")    AS name,
  ANY_VALUE("City/Town")        AS city,
  ANY_VALUE("State")            AS state,
  ANY_VALUE("Telephone Number") AS phone,
  MAX(CASE WHEN "Measure ID"='PSI_03' THEN TRY_CAST(Score AS DOUBLE) END)        AS psi03,
  MAX(CASE WHEN "Measure ID"='PSI_08' THEN TRY_CAST(Score AS DOUBLE) END)        AS psi08,
  MAX(CASE WHEN "Measure ID"='PSI_90' THEN TRY_CAST(Score AS DOUBLE) END)        AS psi90,
  MAX(CASE WHEN "Measure ID"='COMP_HIP_KNEE' THEN TRY_CAST(Score AS DOUBLE) END) AS hip_knee,
  MAX(CASE WHEN "Measure ID"='PSI_03' THEN "Compared to National" END)           AS psi03_vs_national,
  MAX(CASE WHEN "Measure ID"='PSI_08' THEN "Compared to National" END)           AS psi08_vs_national
FROM raw_complications
GROUP BY 1''')

con.execute('''
CREATE OR REPLACE TABLE hac AS
SELECT "Facility ID" AS ccn,
       TRY_CAST("Total HAC Score" AS DOUBLE)        AS hac_score,
       TRY_CAST("PSI 90 Composite Value" AS DOUBLE) AS psi90_value,
       "Payment Reduction"                          AS payment_reduction,
       "Fiscal Year"                                AS fiscal_year
FROM raw_hac''')

print('hospitals in Complications file:', con.execute('SELECT COUNT(*) FROM hosp_psi').fetchone()[0])
print('hospitals scored under HACRP   :', con.execute('SELECT COUNT(*) FROM hac').fetchone()[0])


### Join key

All three files key on `Facility ID`, the six-digit CMS Certification Number.


In [ ]:
q('''SELECT 'hosp_psi' AS tbl, COUNT(*) AS rows, COUNT(DISTINCT ccn) AS distinct_ccn,
            MIN(LENGTH(ccn)) AS min_len, MAX(LENGTH(ccn)) AS max_len FROM hosp_psi
     UNION ALL
     SELECT 'hac', COUNT(*), COUNT(DISTINCT ccn), MIN(LENGTH(ccn)), MAX(LENGTH(ccn)) FROM hac''')


---
## 7. PSI-03 distribution


In [ ]:
q('''SELECT COUNT(psi03)                          AS hospitals_with_psi03,
            ROUND(MEDIAN(psi03), 2)               AS median,
            ROUND(QUANTILE_CONT(psi03, 0.90), 2)  AS p90,
            ROUND(MAX(psi03), 2)                  AS max_rate,
            ROUND(MEDIAN(psi08), 2)               AS psi08_median,
            ROUND(QUANTILE_CONT(psi08, 0.90), 2)  AS psi08_p90
     FROM hosp_psi''')


The maximum is roughly sixteen times the median. That spread is the
reason the next cell matters more than the raw numbers: CMS does not treat
a high rate as meaningful unless the risk-adjusted interval clears the
national rate.


---
## 8. CMS's own comparison to the national rate

This field is CMS's determination, based on whether the hospital's
risk-adjusted confidence interval clears the national rate. It is not
calculated here.


In [ ]:
q('''SELECT psi03_vs_national, COUNT(*) AS hospitals
     FROM hosp_psi WHERE psi03_vs_national IS NOT NULL
     GROUP BY 1 ORDER BY hospitals DESC''')


**Only 49 hospitals in the country are rated better than the national
rate. 178 are rated worse.**

The asymmetry is not evidence that most hospitals underperform. 2,829
hospitals are rated "no different," and 1,692 have no published rate at all
because case volume is too small. The 178 and the 49 are the tails of a
distribution that CMS reports conservatively.


---
## 9. HACRP penalty status

Hospitals in the worst-performing quartile by Total HAC Score have **all**
Medicare inpatient payments reduced by 1 percent — not payments for a single
service line. PSI-90, which contains PSI-03, is one of six component measures.
The other five are CLABSI, CAUTI, surgical site infection, MRSA bacteremia,
and C. difficile infection.


In [ ]:
q('''SELECT payment_reduction, COUNT(*) AS hospitals
     FROM hac GROUP BY 1 ORDER BY hospitals DESC''')


---
## 10. Penalty status by PSI-03 rating


In [ ]:
q('''SELECT h.payment_reduction, p.psi03_vs_national, COUNT(*) AS hospitals
     FROM hosp_psi p JOIN hac h USING(ccn)
     WHERE p.psi03 IS NOT NULL AND h.payment_reduction IS NOT NULL
     GROUP BY 1, 2 ORDER BY 1, hospitals DESC''')


---
## 11. Hospitals both penalized and rated worse on PSI-03

Both fields are published by CMS. This is their intersection, not a derived
score.


In [ ]:
dual = q('''SELECT p.ccn, p.name, p.city, p.state,
            ROUND(p.psi03, 2)     AS psi03,
            ROUND(p.psi08, 3)     AS psi08,
            ROUND(p.psi90, 2)     AS psi90,
            ROUND(h.hac_score, 3) AS hac_score
     FROM hosp_psi p JOIN hac h USING(ccn)
     WHERE h.payment_reduction = 'Yes'
       AND p.psi03_vs_national = 'Worse Than the National Rate'
     ORDER BY p.psi03 DESC''')

print(len(dual), 'hospitals')
dual.head(25)


---
## 12. By state


In [ ]:
by_state = q('''SELECT p.state,
       COUNT(*)                                                             AS hospitals,
       SUM(CASE WHEN h.payment_reduction='Yes' THEN 1 ELSE 0 END)           AS penalized,
       SUM(CASE WHEN p.psi03_vs_national='Worse Than the National Rate'
                THEN 1 ELSE 0 END)                                          AS psi03_worse,
       ROUND(MEDIAN(p.psi03), 2)                                            AS median_psi03
     FROM hosp_psi p JOIN hac h USING(ccn)
     WHERE p.psi03 IS NOT NULL
     GROUP BY 1 ORDER BY penalized DESC''')
by_state.head(20)


State-level differences in reported rates should not be read as
differences in care. Coding completeness varies, and PSI-03 is derived from
administrative discharge data rather than chart review.


---
## 13. Export derived data


In [ ]:
os.makedirs('derived', exist_ok=True)
dual.to_csv('derived/penalized-and-worse-psi03.csv', index=False)
by_state.to_csv('derived/by-state.csv', index=False)
print('written:')
for f in sorted(os.listdir('derived')):
    print(' ', f)


---

## Limitations

**Claims-based measurement.** PSI-03 is derived from administrative discharge
data, not chart review. Published research documents that coding practice
affects the rate — spinal cord injury cases not coded for paralysis are not
excluded and therefore flag as events.

**Suppression.** CMS does not publish a rate where case volume is too small.
1,692 hospitals show "Not Available."

**Differing measurement periods.** HACRP FY2026 uses PSI-90 data through
2024-06-30 and infection data through 2024-12-31. The Complications file
reports a separate period. Penalty status and PSI-03 rate are not measured
over identical windows.

**Cross-sectional.** No time series, no comparison group, no intervention.

**Multiple causation.** Pressure injury arises from nutrition, repositioning
frequency, support surface selection, perfusion, immobility duration, and
mechanical forces during repositioning and transfer. Nothing here isolates any
one of these.

---

Source data is US Government work under 17 U.S.C. §105 and is not subject to
copyright. Code MIT, derived data CC-BY-4.0.
